### 加载数据

In [275]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

In [276]:
iris = load_iris()

In [277]:
X = iris.data
y = iris.target    # 获取目标标签，共 3 个类别

In [278]:
X.shape, y.shape

((150, 4), (150,))

In [279]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=233, stratify=y)    # 将数据划分为训练集和测试集

In [280]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((105, 4), (45, 4), (105,), (45,))

### 超参数

In [281]:
from sklearn.neighbors import KNeighborsClassifier    # 导入 K 近邻分类模型

In [282]:
neigh = KNeighborsClassifier(  # 创建 KNN 分类模型
    n_neighbors=3,       # 使用距离最近的 3 个样本进行预测
    weights="distance",  # 距离越近的邻居权重越大
    p=2)                 # 使用欧氏距离

In [283]:
neigh.fit(X_train, y_train);    # 使用训练集训练模型

In [284]:
neigh.score(X_test, y_test)     # 计算模型在测试集上的准确率

0.9777777777777777

In [285]:
best_score = -1    # 初始化最佳准确率和最佳超参数
best_n = -1
best_weight = ""
best_p = -1
for n in range(1, 20):    # 遍历不同的邻居数量：1～19
    for weight in ["uniform", "distance"]:  # 遍历两种权重计算方式
        for p in range(1, 7):               # 遍历不同的距离参数：1～6
            neigh = KNeighborsClassifier(   # 创建当前超参数组合对应的 KNN 模型
                n_neighbors=n,    # 邻居数量
                weights=weight,   # 邻居权重方式
                p=p)              # 闵可夫斯基距离参数
            neigh.fit(X_train, y_train)            # 使用训练集训练模型
            score = neigh.score(X_test, y_test)    # 计算模型在测试集上的准确率
            if score > best_score:                 # 如果当前准确率更高，则保存当前结果
                best_score = score
                best_n = n
                best_weight = weight
                best_p = p
print("n_neighbors:", best_n)                      # 输出最佳超参数和对应的准确率
print("weights:", best_weight)
print("p:", best_p)
print("score:", best_score)

n_neighbors: 5
weights: uniform
p: 2
score: 1.0


### scikit-learn 超参数搜索

In [286]:
from sklearn.model_selection import GridSearchCV   # 导入网格搜索工具

In [287]:
params = {    # 设置需要搜索的超参数及其候选值
    "n_neighbors": [n for n in range(1, 20)],   # 邻居数量：1～19
    "weights": ["uniform", "distance"],         # 邻居权重方式
    "p": [p for p in range(1, 7)]}              # 距离参数：1～6
grid = GridSearchCV(                    # 创建网格搜索对象
    estimator=KNeighborsClassifier(),   # 使用 KNN 分类模型
    param_grid=params,                  # 指定超参数搜索范围
    n_jobs=-1)                          # 使用全部 CPU 核心并行计算

In [288]:
grid.fit(X_train, y_train);    # 使用训练集进行交叉验证和超参数搜索

In [289]:
grid.best_params_    # 查看最佳超参数组合

{'n_neighbors': 9, 'p': 2, 'weights': 'uniform'}

In [290]:
grid.best_score_    # 查看最佳交叉验证平均得分

np.float64(0.961904761904762)

In [291]:
grid.best_estimator_;    # 查看使用最佳超参数创建的模型

In [292]:
grid.best_estimator_.predict(X_test)    # 使用最佳模型预测测试集

array([2, 2, 0, 1, 1, 1, 2, 0, 2, 0, 0, 1, 0, 2, 1, 1, 0, 2, 2, 1, 0, 1,
       1, 2, 2, 0, 0, 1, 1, 0, 2, 2, 0, 1, 1, 2, 1, 1, 0, 0, 0, 2, 0, 1,
       1])

In [293]:
grid.best_estimator_.score(X_test, y_test)    # 计算最佳模型在测试集上的准确率

0.9555555555555556